# NeuroGraph-X · L3 + L4 · Baseline GNN on real ABIDE (CC200)
Owner: Laiba Afridi. Protocol: memo 0006 (amended 12 Oct) and memo 0007.

**Input:** the `neurographx-code` dataset (repository zip with a COMMIT file).
**Settings:** Accelerator **GPU T4**, Internet **On**.

Run the cells in order. Cross-validation can be run as often as needed. **The FINAL cell is run once.**

## 1. Copy the code into a writable folder

In [ ]:
import glob, os, shutil
code_root = os.path.dirname(glob.glob("/kaggle/input/**/pyproject.toml", recursive=True)[0])
REPO = "/kaggle/working/ngx"
if not os.path.exists(REPO):
    shutil.copytree(code_root, REPO)
os.chdir(REPO)
print("repo:", REPO, "| commit:", open("COMMIT").read().strip() if os.path.exists("COMMIT") else "unknown")

In [ ]:
!pip install -q torch_geometric nilearn==0.14.1 tabulate

## 2. Rebuild the connectomes (Aleesha's scripts, memo 0007). Step 04 downloads about 1 GB and can be re-run if it stops.

In [ ]:
!PYTHONPATH=src python scripts/01_fetch_phenotypic.py

In [ ]:
!PYTHONPATH=src python scripts/04_fetch_all_atlases.py --quiet

In [ ]:
!PYTHONPATH=src python scripts/07_build_connectomes.py
!PYTHONPATH=src python scripts/08_verify_connectomes.py | tail -n 15

In [ ]:
!cat docs/reports/connectome_summary.md | head -n 20

**Check before going on.** The CC200 row must read: participants **861**, nodes **200**, edges **19900**, with flat regions **15**, NaN edges **3969**, mean z **0.008**.
If any number differs, stop and ask.

## 3. Tests (all should pass here, none skipped, because Kaggle has PyTorch)

In [ ]:
!PYTHONPATH=src python -m pytest -q tests/test_leakage.py tests/test_gnn_data.py tests/test_run_logger.py tests/test_inference.py

## 4. Cross-validation (development pool only; the frozen set is not read)

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage cv

Copy the last lines (CV accuracy, majority baseline, balanced accuracy, chosen epochs, run_id) and check them before the final cell.

In [ ]:
CV_RUN_ID = "PASTE_THE_CV_RUN_ID_HERE"

## 5. FINAL: whole development pool, frozen set measured ONCE

In [ ]:
!PYTHONPATH=src python scripts/train_gnn.py --stage final --epochs-from-cv {CV_RUN_ID}

## 6. Inference on one frozen participant (L7, demo step 6)

In [ ]:
import sys; sys.path.insert(0, "src")
import pandas as pd
from neurographx.m3_model.data import load_connectomes, make_get_graph
from neurographx.m3_model.inference import predict
from neurographx.m3_model.run_logger import get_model, get_run
entry = get_model("cc200"); run = get_run(entry["run_id"])
cset = load_connectomes("data/processed/connectomes", "rois_cc200")
get_graph = make_get_graph(cset, run["data_config"]["density"], run["data_config"]["fill_nan"])
sub = int(pd.read_csv("data/frozen/frozen_eval_100.csv").SUB_ID[0])
print(predict(f"{sub:07d}", "cc200", get_graph))

## 7. Evidence: copy the files to download into one folder

In [ ]:
os.makedirs("/kaggle/working/evidence", exist_ok=True)
for f in ["runs/run_log.jsonl", "runs/model_registry.json", "docs/reports/connectome_summary.md"] + glob.glob("checkpoints/cc200_*.pt"):
    shutil.copy(f, "/kaggle/working/evidence/")
print(os.listdir("/kaggle/working/evidence"))
!tail -n 2 runs/run_log.jsonl
!cat runs/model_registry.json

Save Version, then download everything in `/kaggle/working/evidence` from the Output panel.